In [2]:
import torch
import torchnmf
from torchaudio import load
import numpy as np
import matplotlib.pyplot as plt
from scipy.sparse import coo_matrix
import pandas as pd
import scipy.sparse as sp
from sklearn.metrics import explained_variance_score

from nmf import run_nmf

from sklearn.decomposition import non_negative_factorization as SKNMF
from torchnmf.nmf import NMF as torchNMF

import time, random, gc
from termcolor import cprint
from threadpoolctl import threadpool_limits
import os

import muon as mu 
import scanpy as sc


from memory_profiler import memory_usage

torch.set_flush_denormal(True)

torchnmf.__version__

/home/users/ymo/.local/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


'0.3.5'

In [3]:
!lscpu | grep "CPU"
print(torch.cuda.get_device_name())

CPU op-mode(s):        32-bit, 64-bit
CPU(s):                20
On-line CPU(s) list:   0-19
CPU family:            6
Model name:            Intel(R) Xeon(R) CPU E5-2640 v4 @ 2.40GHz
CPU MHz:               1300.048
CPU max MHz:           3400.0000
CPU min MHz:           1200.0000
NUMA node0 CPU(s):     0-9
NUMA node1 CPU(s):     10-19
NVIDIA TITAN Xp


# Load data, function and folder names

In [ ]:
# folder to save the results 
file_name = "/oak/stanford/groups/engreitz/Users/ymo/NMF_re-inplementing/NMF_benchmark_10_runs_random_seeds_D0+D3_7.30.25"

In [4]:
# reading
mdata = mu.read("/oak/stanford/groups/engreitz/Users/ymo/NMF_re-inplementing/250_gene_rawcounts_7.16.25.h5mu")    # returns a MuData object
adata = mdata.mod['rna']
cNNMF = mdata.mod['cNMF']
adata.var_names_make_unique()

/home/users/ymo/.local/lib/python3.9/site-packages/anndata/_core/anndata.py:1820: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/home/users/ymo/.local/lib/python3.9/site-packages/mudata/_core/mudata.py:477: UserWarning: var_names are not unique. To make them unique, call `.var_names_make_unique`.
  warnings.warn(


In [9]:
def sk_cd(k, Snumpy, file_name, name ="", random_state = None):
    
    print("sk_cd started")
    
    #sklearn cd
    start = time.time()
    
    W, H, n_iter = SKNMF(Snumpy, n_components= k, init='random', random_state = random_state, solver="cd", max_iter=500, beta_loss=2.0, tol=1e-4,
                   verbose=True)
        
    
    total_time = (time.time() - start)
    
    err = explained_variance_score(Snumpy,np.dot(W, H))
    
    np.save(f'{file_name}/H_sk_cd_K{k}_cell{Snumpy.shape[0]}_{name}_{i}.npy', H) 
    np.save(f'{file_name}/W_sk_cd_K{k}_cell{Snumpy.shape[0]}_{name}_{i}.npy', W) 
    
    return err, total_time

def nmftorch_halsvar_cuda(k, Scuda ,file_name, name ="" ,random_state = None ):
    
    print("nmftorch_halsvar_cuda started")
    
    #nmftorch mu
    start = time.time()
    torch.cuda.synchronize()

    W, H, err = run_nmf(Scuda, n_components= k, init='random', algo="halsvar", beta_loss= 2.0, tol=1e-4, random_state = random_state,
                  use_gpu = True )
    
    torch.cuda.synchronize()
    total_time = (time.time() - start)
    
    err = explained_variance_score(Scuda.detach().cpu().numpy(),np.dot(W, H))

    
    np.save(f'{file_name}/H_nmftorch_cuda_K{k}_cell{Scuda.shape[0]}_{name}_{i}.npy', H) 
    np.save(f'{file_name}/W_nmftorch_cuda_K{k}_cell{Scuda.shape[0]}_{name}_{i}.npy', W) 
    
    
    return err, total_time
   

# 10 run for sk_cd and nmftorch_halsv at 10k cells

In [14]:
adata_sub = sc.pp.subsample(
    adata,
    n_obs=10000,              
    copy=True            
)

Snumpy = adata_sub.layers["raw_counts"]
Storch = torch.from_numpy(Snumpy).float()
Scuda = Storch.cuda()


In [15]:
for i in range(1,11):

    peak_MiB, metrics = memory_usage((sk_cd,(10,Snumpy,file_name)),max_usage=True,retval=True)
    peak_MiB, metrics = memory_usage((nmftorch_halsvar_cuda,(10,Scuda,file_name,i)),max_usage=True,retval=True)


sk_cd started
violation: 1.0
violation: 1.5852047184807843
violation: 0.6045024687058349
violation: 0.3564615655373679
violation: 0.2796816740651312
violation: 0.21083897047056918
violation: 0.18037130076137647
violation: 0.16515485670969976
violation: 0.14769594743950173
violation: 0.12794216154560914
violation: 0.1076853466772659
violation: 0.08966191007866695
violation: 0.07580916685493991
violation: 0.06527695082475883
violation: 0.05733929512431551
violation: 0.051257511415765435
violation: 0.046644621917079736
violation: 0.04308808894246476
violation: 0.040578073594318434
violation: 0.03908484115916922
violation: 0.03842730799473008
violation: 0.03847748961420741
violation: 0.0389308944156047
violation: 0.039646906061747415
violation: 0.04050697978858174
violation: 0.041210629726567256
violation: 0.04159381344051763
violation: 0.04148172348653868
violation: 0.04084108152081249
violation: 0.03983037649003459
violation: 0.03850892082084387
violation: 0.0370106230600928
violation: 0

/home/users/ymo/.local/lib/python3.9/site-packages/sklearn/decomposition/_nmf.py:1742: ConvergenceWarning: Maximum number of iterations 500 reached. Increase it to improve convergence.
  warnings.warn(


nmftorch_halsvar_cuda started
Use GPU mode.
 niter=10, loss=2704.0710049848913.
 niter=20, loss=2702.3134533210614.
 niter=30, loss=2702.2636806943915.
    Converged after 30 iteration(s).
sk_cd started
violation: 1.0
violation: 1.2764726131809883
violation: 0.6384124964745641
violation: 0.3690375567751055
violation: 0.2618158456911673
violation: 0.19481043485158228
violation: 0.16008560442698547
violation: 0.1377623356685299
violation: 0.12034977348364848
violation: 0.10502023738167929
violation: 0.0930950778907537
violation: 0.08272905570532087
violation: 0.07322857273591274
violation: 0.06446449833192759
violation: 0.056219573660938126
violation: 0.048827713787434904
violation: 0.042633935272383794
violation: 0.03737315731540978
violation: 0.03298962788424868
violation: 0.02939057116207514
violation: 0.02644774981765838
violation: 0.024049786216387675
violation: 0.022098806388562382
violation: 0.020614311110231893
violation: 0.019311169458076215
violation: 0.01835228489702916
violat

# 10 runs for individual package and dates 

In [5]:
adata_day0 = adata[adata.obs["sample"] == "D0"]
adata_day3 = adata[adata.obs["sample"] == "sample_D3"]

In [6]:
Snumpy_day0 = adata_day0.X.toarray()
Storch_day0 = torch.from_numpy(Snumpy_day0).float()

Snumpy_day3 = adata_day3.X.toarray()
Storch_day3 = torch.from_numpy(Snumpy_day3).float()

In [10]:

for i in range(1,11):

    peak_MiB, metrics = memory_usage((sk_cd,(10,Snumpy_day0,file_name,"Day0")),max_usage=True,retval=True)
    
    peak_MiB, metrics = memory_usage((sk_cd,(10,Snumpy_day3,file_name,"Day3")),max_usage=True,retval=True)
    
    peak_MiB, metrics = memory_usage((nmftorch_halsvar_cuda,(10, Storch_day0, file_name, "Day0", i)),max_usage=True,retval=True)
    
    peak_MiB, metrics = memory_usage((nmftorch_halsvar_cuda,(10, Storch_day3,  file_name, "Day3", i+10 )),max_usage=True,retval=True)

sk_cd started
violation: 1.0
violation: 0.9029955057864132
violation: 0.4178428739563972
violation: 0.2941253705037565
violation: 0.23561470394590503
violation: 0.20191839397906142
violation: 0.17630668752962575
violation: 0.15510635376440732
violation: 0.13429341799631733
violation: 0.11415133626873028
violation: 0.09555416837771837
violation: 0.08036324636775824
violation: 0.06794735707124482
violation: 0.05774215785875034
violation: 0.049768304299123836
violation: 0.04339187271763193
violation: 0.03834896421437566
violation: 0.03438723304408061
violation: 0.030974033407533994
violation: 0.02817274662330758
violation: 0.025735089768281778
violation: 0.023607172412786624
violation: 0.021680109395033332
violation: 0.0200679812113758
violation: 0.01859804063669394
violation: 0.017271562408620295
violation: 0.01612531703647622
violation: 0.0150767595284058
violation: 0.014055940640335846
violation: 0.013205823461327853
violation: 0.012402729055648126
violation: 0.011675853851825279
viola

/home/users/ymo/.local/lib/python3.9/site-packages/sklearn/decomposition/_nmf.py:1742: ConvergenceWarning: Maximum number of iterations 500 reached. Increase it to improve convergence.
  warnings.warn(


sk_cd started
violation: 1.0
violation: 2.9520627052719544
violation: 0.674985968231383
violation: 0.40614371413762007
violation: 0.32007991318373785
violation: 0.2708138271825864
violation: 0.23398158828372234
violation: 0.20591499727148768
violation: 0.18071707801500825
violation: 0.15910069953333222
violation: 0.14263017253461835
violation: 0.13096926919709115
violation: 0.12260846290897692
violation: 0.11617721887459036
violation: 0.11088869307831234
violation: 0.10666626733740867
violation: 0.10257436709057015
violation: 0.09843623995562527
violation: 0.09454772817935413
violation: 0.09057115189839147
violation: 0.08682672905327359
violation: 0.08339252559132415
violation: 0.0799999854581991
violation: 0.07560822579208228
violation: 0.07240665265150707
violation: 0.06849931245082251
violation: 0.06514132085658221
violation: 0.06205300802925537
violation: 0.05918931897880802
violation: 0.05647480153970299
violation: 0.053790174986194914
violation: 0.05129170433456441
violation: 0.0

/home/users/ymo/.local/lib/python3.9/site-packages/sklearn/decomposition/_nmf.py:1742: ConvergenceWarning: Maximum number of iterations 500 reached. Increase it to improve convergence.
  warnings.warn(


sk_cd started
violation: 1.0
violation: 2.888356011882116
violation: 0.8139802192917689
violation: 0.4626129156554617
violation: 0.3445170779493416
violation: 0.27462161964798004
violation: 0.22867823707584026
violation: 0.19538143250286552
violation: 0.16916084199073828
violation: 0.1482340002041437
violation: 0.1304183509461036
violation: 0.11478086499730347
violation: 0.1005638524175228
violation: 0.08864177659512314
violation: 0.07918359254967257
violation: 0.07169293364126886
violation: 0.06581230571195461
violation: 0.05945266172454891
violation: 0.05490752889706202
violation: 0.05215187094879838
violation: 0.050390840680057426
violation: 0.04908603220415509
violation: 0.048143992530908106
violation: 0.047161301424267256
violation: 0.04651982111643946
violation: 0.04537344062659413
violation: 0.044398965650759147
violation: 0.043576800942170875
violation: 0.04280915520281584
violation: 0.04220674722036505
violation: 0.041766314388087
violation: 0.04136603806071699
violation: 0.04

# runs for commbined days and individual packages

In [5]:
mask = adata.obs["sample"].isin(["D0", "sample_D3"])    

adata_d0_d3 = adata[mask].copy()

print(adata_d0_d3) 

Snumpy  = adata_d0_d3.X.toarray()
Storch = torch.from_numpy(Snumpy).float()

AnnData object with n_obs × n_vars = 61685 × 5451
    obs: 'sample', 'species', 'gene_count', 'tscp_count', 'mread_count', 'leiden', 'n_counts'
    obsm: 'X_pca', 'X_umap'
    layers: 'norm10k', 'raw_counts'


In [ ]:
for i in range(1,11):

    peak_MiB, metrics = memory_usage((sk_cd,(10,Storch, file_name, "D0_3",)),max_usage=True,retval=True)
        
    peak_MiB, metrics = memory_usage((nmftorch_halsvar_cuda,(10, Storch, file_name, "D0_3", i)),max_usage=True,retval=True)
    

sk_cd started
violation: 1.0
violation: 2.209694658646592
violation: 0.6524844349560562
violation: 0.43775270769546193
violation: 0.3270378867799444
violation: 0.2631712824660356
violation: 0.2206480623357419
violation: 0.17782776674738793
violation: 0.13900202696147632
violation: 0.1099823449328741
violation: 0.08999845459679494
violation: 0.07674958982121807
violation: 0.06739063635266611
violation: 0.06016045094263252
violation: 0.05428061828272288
violation: 0.04965940817816923
violation: 0.04580310530746349
violation: 0.042731179419108825
violation: 0.040164096767975485
violation: 0.03802304554249119
violation: 0.036238615778543225
violation: 0.034759818028323
violation: 0.033528154915433865
violation: 0.03247841558739406
violation: 0.0315968747972856
violation: 0.030851581859162036
violation: 0.030203517972517437
violation: 0.02964493077761402
violation: 0.029170224286054706
violation: 0.028822356707057448
violation: 0.028599595831169528
violation: 0.028527382807295353
violation: